In [1]:
import sys
import os

# Get the current working directory
cwd = os.getcwd()
print(f"Current Working Directory: {cwd}")

# Define the path to the 'mapelite' folder
# We assume the notebook is running from the root 'Quality-Diversity-...' folder
mapelite_path = os.path.join(cwd, 'mapelite')

# Add it to the system path so Python can find config.py, utils.py, etc.
if mapelite_path not in sys.path:
    sys.path.append(mapelite_path)
    print(f"Added '{mapelite_path}' to sys.path")

Current Working Directory: d:\dev\Quality-Diversity-for-Racing-Track-Design
Added 'd:\dev\Quality-Diversity-for-Racing-Track-Design\mapelite' to sys.path


In [2]:
import numpy as np
import glob
import pickle
import random
import json


from ribs.archives import ProximityArchive, AddStatus
from ribs.schedulers import Scheduler

from dask.distributed import Client, LocalCluster, as_completed

from utils import array_to_solution
from mapelite.evaluator import EvaluatorNoveltySearch
from emitter import CustomEmitter

In [3]:
from mapelite.config import (
    SOLUTION_DIM,
    BATCH_SIZE,
    CHECKPOINT_DIR,
    HEATMAP_DIR,
    CHECKPOINT_EVERY,
    INVALID_SCORE,
    ITERATIONS
)
STATS_DIR = "data/stats/"
EMBEDDING_DIM = 32
DEFAULT_THRESHOLD = 5
SEED = 67

random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
np.random.seed(SEED)



In [4]:
# --- Initialize directories ---
# Create directories if they don't exist
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(HEATMAP_DIR, exist_ok=True)

In [5]:
# --- DASK SETUP ---
print("Setting up Dask LocalCluster...")
cluster = LocalCluster(processes=True, n_workers=BATCH_SIZE, threads_per_worker=1)
client = Client(cluster)
print(f"Dask Dashboard link: {client.dashboard_link}")

Setting up Dask LocalCluster...
Dask Dashboard link: http://127.0.0.1:8787/status


In [6]:
archive = ProximityArchive(
    solution_dim=SOLUTION_DIM,
    measure_dim=EMBEDDING_DIM,
    k_neighbors=15,
    novelty_threshold=DEFAULT_THRESHOLD,
    seed=SEED,
    local_competition=True
)

emitter = CustomEmitter(
    archive,
    solution_dim=SOLUTION_DIM,
    batch_size=BATCH_SIZE,
    # Last element (ID) can be arbitrarily large (ID=iteration + fractional_part)
    bounds=[(0, 600)] * (SOLUTION_DIM - 1) + [(0, float("inf"))]
)

scheduler = Scheduler(archive, [emitter])

evaluator = EvaluatorNoveltySearch()

# Scatter the evaluator to all workers once (avoids re-serializing the model every submit)
evaluator_future = client.scatter(evaluator, broadcast=True)


Loading model from mapelite/embeddings/models/model_metrics_VAE/model_metrics_VAE_latent32.pth...
Model loaded with latent_dim=32


In [7]:
# # --- Pilot: measure pairwise distances in embedding space ---
# from scipy.spatial.distance import pdist

# PILOT_ITERS = 5  # number of batches to sample
# pilot_measures = []

# for _ in range(PILOT_ITERS):
#     sols = scheduler.ask()
#     sol_dicts = [array_to_solution(sol) for sol in sols]

#     futs = [client.submit(_eval_on_worker, evaluator_future, sol) for sol in sol_dicts]
#     gathered = [f.result() for f in as_completed(futs)]

#     for sol_id, ok, msg, score, desc in gathered:
#         if ok:
#             pilot_measures.append(desc)

#     # Still need to tell the scheduler so it doesn't break
#     objs, descs = zip(*[(score if ok else INVALID_SCORE, desc)
#                         for sol_id, ok, msg, score, desc in gathered])
#     scheduler.tell(list(objs), list(descs))

# pilot_measures = np.array(pilot_measures)
# dists = pdist(pilot_measures)

# print(f"Pilot: {len(pilot_measures)} valid embeddings collected")
# print(f"Pairwise distances — min={dists.min():.4f}, "
#       f"25th={np.percentile(dists, 25):.4f}, "
#       f"median={np.median(dists):.4f}, "
#       f"75th={np.percentile(dists, 75):.4f}, "
#       f"max={dists.max():.4f}, "
#       f"mean={dists.mean():.4f}")
# print(f"\nSuggested novelty_threshold range: {np.percentile(dists, 10):.4f} – {np.median(dists) * 0.5:.4f}")

In [8]:
# --------------------------------------------------------------
# Resume from latest pickle checkpoint and stats if available
# --------------------------------------------------------------
checkpoints = sorted(glob.glob(f"{CHECKPOINT_DIR}checkpoint_*.pkl"))
start_iter = 0
global_best_score = INVALID_SCORE
global_best_id = None
stats = []

if checkpoints:
    latest_ckpt = checkpoints[-1]
    with open(latest_ckpt, "rb") as f:
        state = pickle.load(f)
    scheduler           = state["scheduler"]
    # Get the latest archive instance from the scheduler
    archive             = scheduler.archive
    start_iter          = state["iteration"]
    global_best_score   = state["global_best_score"]
    global_best_id      = state["global_best_id"]
    print(f"[Resume] Loaded {latest_ckpt}, resuming from iteration {start_iter+1}")
    
    # Resume also stats
    stats_file = f"{STATS_DIR}stats.pkl"
    if os.path.exists(stats_file):
        with open(stats_file, "rb") as f:
            stats = pickle.load(f)
    print(f"[Resume] Resumed stats with {len(stats)} entries")
   
else:
    print("[Resume] No checkpoint found — starting fresh.")

[Resume] No checkpoint found — starting fresh.


In [ ]:



def _eval_on_worker(evaluator, sol):
    """Thin wrapper so Dask receives the evaluator as an already-scattered future."""
    return evaluator.evaluate(sol)

def run_novelty_search(total_iters, start_iter=0):
    global global_best_score, global_best_id
    
    for i in range(start_iter, total_iters):
        
        # Ask the scheduler for a batch of solutions to evaluate
        sols = scheduler.ask()
        sol_dicts = [array_to_solution(sol) for sol in sols]
        
        # submit evaluation tasks to Dask and wait for results
        futs = [client.submit(_eval_on_worker, evaluator_future, sol) for sol in sol_dicts]
        gathered = [f.result() for f in as_completed(futs)]
        
        obj_list, clean_solutions = [], []
        for sol_id, ok, msg, score, desc in gathered:
            
            if not ok:
                print(f"Warning: clamping bad score for ID={sol_id} ({msg})")
                score = INVALID_SCORE
            else:
                print(f"Solution ID={sol_id} evaluated with score={score:.2f}")
                if score > global_best_score:
                    global_best_score = score
                    global_best_id = sol_id
            
            clean_solutions.append((score, desc))
            obj_list.append(score)
        
        # Tell the scheduler the results
        obj_batch, meas_batch = zip(*clean_solutions)
        
        new_elites_count = 0
        sub_elites_count = 0
        original_add = archive.add
        
        def tracked_add(*args, **kwargs):
            nonlocal new_elites_count, sub_elites_count
            res = original_add(*args, **kwargs)
            
            # Extract statuses (format varies slightly depending on pyribs version)
            statuses = None
            if isinstance(res, tuple):
                statuses = res[0]
            elif hasattr(res, 'status'):
                statuses = res.status
            elif isinstance(res, dict) and 'status' in res:
                statuses = res['status']
                
            if statuses is not None:
                arr = np.asarray(statuses)
                # Count insertions based on AddStatus mapping (NEW=2, IMPROVE_EXISTING=1)
                new_elites_count += int(np.sum(arr == AddStatus.NEW))
                sub_elites_count += int(np.sum(arr == AddStatus.IMPROVE_EXISTING))
                
            return res
        
        # Apply the temporary patch
        archive.add = tracked_add
        
        try:
            # Tell the scheduler the results (which uses our patched archive.add internally)
            scheduler.tell(list(obj_batch), list(meas_batch))
        finally:
            # Remove the patch immediately so the object remains picklable later
            if 'add' in archive.__dict__:
                del archive.__dict__['add']
        
        # Log statistics
        batch_best = max(obj_list) if obj_list else INVALID_SCORE
        print(f"Iteration {i+1} ended. Best in batch = {batch_best:.2f}")
        print(f"Global best so far: {global_best_score:.2f} (ID={global_best_id})")
        
        print(f"Archive Updates: {new_elites_count} new elites inserted, {sub_elites_count} elites substituted.")
        
        # print archive stats at each iteration
        data_archive = archive.data()
        if data_archive:
            arch_obj = data_archive["objective"]
            valid = arch_obj != INVALID_SCORE
            mean_val = np.mean(arch_obj[valid]) if np.any(valid) else 0.0
            best_val = np.max(arch_obj[valid]) if np.any(valid) else 0.0
            elites = archive.stats.num_elites
            print(f"Archive size={elites}, mean={mean_val:.2f}, best={best_val:.2f}")
        else:
            print("Archive still empty")
            
        # Checkpoint
        if (i + 1) % CHECKPOINT_EVERY == 0:
            ckpt_name = f"{CHECKPOINT_DIR}checkpoint_{i+1:04d}.pkl"
            with open(ckpt_name, "wb") as f:
                pickle.dump({
                    "scheduler": scheduler,
                    "iteration": i+1,
                    "global_best_score": global_best_score,
                    "global_best_id": global_best_id
                }, f)
            print(f"[Checkpoint] Saved {ckpt_name}")
            # also save stats to a separate json file for storage
            stats_dir= f"{STATS_DIR}stats.pkl"
            with open(stats_dir, "wb") as f:
                pickle.dump(stats, f)
                
        stats.append({
            "iteration": i,
            "Archive size": elites,
            "iteration_best": batch_best,
            "global_best_score": global_best_score,
            "new_elites": new_elites_count,
            "substituted_elites": sub_elites_count
        })

In [12]:
# Run main loop
run_novelty_search(ITERATIONS, start_iter)

Emitter.ask() called for iteration 3
Solution ID=2.270592228132468 evaluated with score=19.00
Solution ID=2.761546025871752 evaluated with score=8.00
Solution ID=2.314061612610119 evaluated with score=7.75
Solution ID=2.1230509366033923 evaluated with score=2.25
Solution ID=2.5863026715917536 evaluated with score=11.75
Solution ID=2.673990527143581 evaluated with score=12.25
Solution ID=2.807224722980778 evaluated with score=12.00
Solution ID=2.6490361846969908 evaluated with score=9.25
Solution ID=2.8067094589252077 evaluated with score=11.00
Solution ID=2.3874631351251727 evaluated with score=20.25
Iteration 1 ended. Best in batch = 20.25
Global best so far: 20.25 (ID=2.3874631351251727)
Archive Updates: 6 new elites inserted, 4 elites substituted.


TypeError: object of type 'int' has no len()

In [ ]:
# --- Visualize stats after running the algorithm ---
import matplotlib.pyplot as plt

iterations      = [s["iteration"] + 1 for s in stats]
archive_sizes   = [s["Archive size"] for s in stats]
iter_best       = [s["iteration_best"] for s in stats]
global_best     = [s["global_best_score"] for s in stats]
new_elites      = [s["new_elites"] for s in stats]
sub_elites      = [s["substituted_elites"] for s in stats]

fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharex=True)
fig.suptitle("Novelty Search — Run Statistics", fontsize=16, fontweight="bold")

# 1. Archive size over iterations
ax = axes[0, 0]
ax.plot(iterations, archive_sizes, color="tab:blue", linewidth=1.5)
ax.set_ylabel("Archive Size")
ax.set_title("Archive Growth")
ax.grid(True, alpha=0.3)

# 2. Best fitness: per-iteration vs global
ax = axes[0, 1]
ax.plot(iterations, iter_best, label="Iteration Best", color="tab:orange", alpha=0.6, linewidth=1)
ax.plot(iterations, global_best, label="Global Best", color="tab:red", linewidth=2)
ax.set_ylabel("Fitness Score")
ax.set_title("Fitness Progress")
ax.legend()
ax.grid(True, alpha=0.3)

# 3. New elites & substituted elites per iteration (stacked bar)
ax = axes[1, 0]
bar_width = max(1, len(iterations) // 200)
ax.bar(iterations, new_elites, width=bar_width, label="New Elites", color="tab:green", alpha=0.8)
ax.bar(iterations, sub_elites, width=bar_width, bottom=new_elites, label="Substituted Elites", color="tab:purple", alpha=0.8)
ax.set_xlabel("Iteration")
ax.set_ylabel("Count")
ax.set_title("Elite Insertions per Iteration")
ax.legend()
ax.grid(True, alpha=0.3)

# 4. Cumulative elites inserted
ax = axes[1, 1]
cum_new = np.cumsum(new_elites)
cum_sub = np.cumsum(sub_elites)
ax.plot(iterations, cum_new, label="Cumulative New", color="tab:green", linewidth=1.5)
ax.plot(iterations, cum_sub, label="Cumulative Substituted", color="tab:purple", linewidth=1.5)
ax.plot(iterations, cum_new + cum_sub, label="Cumulative Total", color="tab:blue", linewidth=2, linestyle="--")
ax.set_xlabel("Iteration")
ax.set_ylabel("Cumulative Count")
ax.set_title("Cumulative Elite Insertions")
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

# --- Summary table ---
print(f"\n{'='*50}")
print(f"  Novelty Search Summary")
print(f"{'='*50}")
print(f"  Total iterations:        {len(stats)}")
print(f"  Final archive size:      {archive_sizes[-1]}")
print(f"  Global best fitness:     {global_best[-1]:.4f}")
print(f"  Total new elites:        {sum(new_elites)}")
print(f"  Total substituted:       {sum(sub_elites)}")
print(f"  Avg new elites/iter:     {np.mean(new_elites):.2f}")
print(f"  Avg substituted/iter:    {np.mean(sub_elites):.2f}")
print(f"{'='*50}")